In [2]:
!pip install s3tokenizer

  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 37.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.8/566.8 kB 42.0 MB/s eta 0:00:00
Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gruut 2.2.3 requires numpy<2.0.0,>=1.19.0, but you have numpy 2.4.6 which is incompatible.

[notice] A new rele

In [ ]:
from datetime import datetime
import os
import torch
from torch import nn
import numpy as np
from datasets import load_dataset
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig # type: ignore
import evaluate
from torch.utils.tensorboard import SummaryWriter
from transformers import EarlyStoppingCallback
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('punkt_tab')
import librosa
from IPython.display import Audio
import soundfile as sf
import s3tokenizer
from s3tokenizer import S3TokenizerV2
import pandas as pd
from huggingface_hub import snapshot_download
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Callable
import inspect
import torch.nn.functional as F


In [2]:
AUDIO_PATH = "../../data/dialogs-ru-emotional-conversations"

In [64]:
snapshot_download(
    "langswap/dialogs-ru-emotional-conversations",
    local_dir=AUDIO_PATH,
    repo_type="dataset"
)

Fetching ... files: 0it [00:00, ?it/s]

'/Users/philippplotnikov/WorkingSpace/University/MachineLearning/Nlp/povaryoshka-rag-assistant/data/dialogs-ru-emotional-conversations'

In [88]:
train_df = pd.read_csv(f"{AUDIO_PATH}/train.csv", sep="|")
train_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,wavs/masha_dima_part1_1.wav,D,"Привет, Маша!",happy,счастье,"Прив+ет, М+аша!",1.800000
1,wavs/masha_dima_part1_2.wav,M,"Привет, Дима! Как дела?",happy,счастье,"Прив+ет, Д+има! К+ак дел+а?",1.998729
2,wavs/masha_dima_part1_3.wav,D,"Да, погодка сегодня... невероятная. А ты тоже ...",happy,счастье,"Д+а, пог+одка сег+одня... неверо+ятная. +А т+ы...",5.461313
3,wavs/masha_dima_part1_4.wav,M,"О, да, весна - моё любимое время года! Всё вок...",happy,счастье,"+О, д+а, весн+а - мо+ё люб+имое вр+емя г+ода! ...",6.358125
4,wavs/masha_dima_part1_5.wav,D,"Согласен, я тоже обожаю весну. Особенно нравит...",happy,счастье,"Согл+асен, +я т+оже обож+аю весн+у. Ос+обенно ...",8.769063
...,...,...,...,...,...,...,...
11423,wavs/masha_sveta_part10_2154.wav,M,Интересно! А ты любишь петь новогодние песни?,happy,счастье,Интер+есно! +А т+ы л+юбишь п+еть новог+одние п...,4.080375
11424,wavs/masha_sveta_part10_2155.wav,S,Очень! Пение новогодних песен - это всегда вес...,happy,счастье,+Очень! П+ение новог+одних п+есен - +это всегд...,5.846271
11425,wavs/masha_sveta_part10_2156.wav,M,Согласна! А ты когда-нибудь участвовала в ново...,happy,счастье,Согл+асна! +А т+ы когд+а-ниб+удь уч+аствовала ...,3.900604
11426,wavs/masha_sveta_part10_2157.wav,S,"Да, это одно из моих любимых занятий на праздн...",happy,счастье,"Д+а, +это одн+о +из мо+их люб+имых зан+ятий н+...",5.305583


In [102]:
val_df = pd.read_csv(f"{AUDIO_PATH}/val.csv", sep="|")
val_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,wavs/masha_dima_part1_119.wav,D,"О, это знакомо. Постоянные отчёты, проверки. К...",angry,злость,"+О, +это знак+омо. Посто+янные отч+ёты, пров+е...",17.231521
1,wavs/masha_dima_part1_107.wav,D,Всё это настолько несправедливо! Хочется крича...,angry,злость,Вс+ё +это наст+олько несправедл+иво! Х+очется ...,11.926854
2,wavs/masha_dima_part1_121.wav,D,Это просто кошмар. С такими руководителями раб...,angry,злость,+Это пр+осто кошм+ар. С так+ими руковод+ителям...,12.660479
3,wavs/masha_dima_part12_23.wav,D,Твоя работа всегда казалась мне такой далёкой ...,angry,злость,Тво+я раб+ота всегд+а каз+алась мн+е так+ой да...,14.474083
4,wavs/masha_dima_part9_255.wav,D,"Да хватит уже, Маша! Я не могу всегда быть иде...",angry,злость,"Д+а хв+атит +уже, М+аша! +Я н+е мог+у всегд+а ...",7.682875
...,...,...,...,...,...,...,...
175,wavs/masha_sveta_part6_50.wav,S,Звучит круто! Надо будет попробовать.,surprise,удивление,Звуч+ит кр+уто! Н+адо б+удет попр+обовать.,3.044021
176,wavs/masha_sveta_part10_1330.wav,S,"Маша, а как обстоят дела с кишечными инфекциям...",neutral,нейтрально,"М+аша, +а к+ак обсто+ят дел+а с киш+ечными инф...",3.960708
177,wavs/masha_sveta_part2_351.wav,S,Всё это подтвердило его решение попробовать по...,neutral,нейтрально,Вс+ё +это подтверд+ило ег+о реш+ение попр+обов...,4.404396
178,wavs/masha_sveta_part5_211.wav,S,"Зарайск, улица Ленина, дом четыреста.",arrogance,высокомерие,"Зар+айск, +улица Л+енина, д+ом чет+ыреста.",4.954458


In [103]:
print(train_df['duration'].sum() / 3600)
print(val_df['duration'].sum() / 3600)

16.861661756503526
0.3044736442192618


In [105]:
print(train_df['emotion'].unique())
print("="*6)
print(val_df['emotion'].unique())

['happy' 'surprise' 'neutral' 'laughing' 'sad' 'angry' 'disgust' 'fear'
 'whisper' 'arrogance' 'sgorogovrki' 'poem']
['angry' 'happy' 'arrogance' 'neutral' 'disgust' 'surprise' 'fear'
 'laughing' 'sad' 'whisper' 'poem' 'sgorogovrki']


In [4]:
target_sr = 16000
TARGET_WAV_FOLDER = "wavs-16KHz"

In [ ]:
for row in tqdm(train_df.itertuples()):
    current_audio_path = str(row.audio_path)
    full_path = f'{AUDIO_PATH}/{current_audio_path}'
    if not os.path.isfile(full_path):
        train_df.at[row.Index, "audio_path"] = None
        continue
    original_waveform, original_sr = sf.read(full_path)
    resampled_waveform = librosa.resample(original_waveform, orig_sr=original_sr, target_sr=target_sr)
    new_audio_path = f"{AUDIO_PATH}/{TARGET_WAV_FOLDER}/{os.path.basename(current_audio_path)}"
    sf.write(new_audio_path, resampled_waveform, target_sr)
    train_df.at[row.Index, "audio_path"] = new_audio_path

11428it [00:31, 360.08it/s]


In [112]:
for row in tqdm(val_df.itertuples()):
    current_audio_path = str(row.audio_path)
    full_path = f'{AUDIO_PATH}/{current_audio_path}'
    if not os.path.isfile(full_path):
        val_df.at[row.Index, "audio_path"] = None
        continue
    original_waveform, original_sr = sf.read(full_path)
    resampled_waveform = librosa.resample(original_waveform, orig_sr=original_sr, target_sr=target_sr)
    new_audio_path = f"{AUDIO_PATH}/{TARGET_WAV_FOLDER}/{os.path.basename(current_audio_path)}"
    sf.write(new_audio_path, resampled_waveform, target_sr)
    val_df.at[row.Index, "audio_path"] = new_audio_path

180it [00:00, 331.81it/s]


In [ ]:
train_result = train_df[train_df['audio_path'].isna()]
print(len(train_result))
train_result

1742


,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
38,None,M,"Ха-ха-ха, ничего страшного. Я как раз давно хо...",laughing,смех,"Х+а-х+а-х+а, ничег+о стр+ашного. +Я к+ак р+аз ...",11.581458
39,None,M,"Ха-ха-ха, представляю: учёный спасает мир, сме...",laughing,смех,"Х+а-х+а-х+а, представл+яю: уч+ёный спас+ает м+...",9.707917
42,None,M,"О, расскажи! Что готовил?",happy,счастье,"+О, расскаж+и! Чт+о гот+овил?",2.590000
47,None,M,Конечно! А ты попробуешь угостить меня своей с...,laughing,смех,Кон+ечно! +А т+ы попр+обуешь угост+ить мен+я с...,4.937646
48,None,D,"Ха-ха-ха-ха, договорились!",laughing,смех,"Х+а-х+а-х+а-х+а, договор+ились!",3.012313
...,...,...,...,...,...,...,...
11417,None,S,Очень! Готовка к праздникам - это всегда особе...,happy,счастье,+Очень! Гот+овка к пр+аздникам - +это всегд+а ...,6.218021
11419,None,M,Здорово! А ты когда-нибудь пробовала готовить ...,happy,счастье,Здор+ово! +А т+ы когд+а-ниб+удь пр+обовала гот...,3.995354
11420,None,S,"Да, всегда стараюсь добавить что-то новое в пр...",happy,счастье,"Д+а, всегд+а стар+аюсь доб+авить чт+о-то н+ово...",6.429396
11421,None,M,Согласна! А ты когда-нибудь устраивала новогод...,happy,счастье,Согл+асна! +А т+ы когд+а-ниб+удь устр+аивала н...,3.485313


In [113]:
val_result = val_df[val_df['audio_path'].isna()]
print(len(val_result))
val_result

32


,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,None,D,"О, это знакомо. Постоянные отчёты, проверки. К...",angry,злость,"+О, +это знак+омо. Посто+янные отч+ёты, пров+е...",17.231521
4,None,D,"Да хватит уже, Маша! Я не могу всегда быть иде...",angry,злость,"Д+а хв+атит +уже, М+аша! +Я н+е мог+у всегд+а ...",7.682875
14,None,D,"Ну, стоимость у него, конечно, не маленькая - ...",disgust,отвращение,"Н+у, ст+оимость +у нег+о, кон+ечно, н+е м+ален...",19.581542
17,None,D,"Именно так я чувствую себя сейчас. Каждый раз,...",fear,страх,+Именно т+ак +я ч+увствую себ+я сейч+ас. К+ажд...,18.792167
18,None,D,Звучит сложно. Какие сложности ты встретила на...,fear,страх,Звуч+ит сл+ожно. Как+ие сл+ожности т+ы встр+ет...,4.340000
21,None,D,"Маша, ты знаешь такую считалочку, которую мы ч...",happy,счастье,"М+аша, т+ы зн+аешь так+ую счит+алочку, кот+ору...",5.570000
26,None,D,"Конечно, они же тоже люди. Только их приключен...",laughing,смех,"Кон+ечно, он+и ж+е т+оже л+юди. Т+олько +их пр...",9.892500
44,None,D,"Ты всегда знала, как найти нужные слова. Когда...",sad,грусть,"Т+ы всегд+а зн+ала, к+ак найт+и н+ужные слов+а...",13.840833
50,None,M,Я беру с собой минималистичные украшения и фун...,neutral,нейтрально,+Я бер+у с соб+ой минималист+ичные украш+ения ...,3.285479
73,None,M,"Как показал вопрос SuperJob, каждый второй рос...",neutral,нейтрально,"К+ак показ+ал вопр+ос SuperJob, к+аждый вт+оро...",6.424563


In [ ]:
train_df.drop(train_result.index, inplace=True)

In [ ]:
print(len(train_df))

In [ ]:
val_df.drop(val_result.index, inplace=True)

In [115]:
print(len(val_df))

148


In [92]:
train_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,../../data/dialogs-ru-emotional-conversations/...,D,"Привет, Маша!",happy,счастье,"Прив+ет, М+аша!",1.800000
1,../../data/dialogs-ru-emotional-conversations/...,M,"Привет, Дима! Как дела?",happy,счастье,"Прив+ет, Д+има! К+ак дел+а?",1.998729
2,../../data/dialogs-ru-emotional-conversations/...,D,"Да, погодка сегодня... невероятная. А ты тоже ...",happy,счастье,"Д+а, пог+одка сег+одня... неверо+ятная. +А т+ы...",5.461313
3,../../data/dialogs-ru-emotional-conversations/...,M,"О, да, весна - моё любимое время года! Всё вок...",happy,счастье,"+О, д+а, весн+а - мо+ё люб+имое вр+емя г+ода! ...",6.358125
4,../../data/dialogs-ru-emotional-conversations/...,D,"Согласен, я тоже обожаю весну. Особенно нравит...",happy,счастье,"Согл+асен, +я т+оже обож+аю весн+у. Ос+обенно ...",8.769063
...,...,...,...,...,...,...,...
11423,../../data/dialogs-ru-emotional-conversations/...,M,Интересно! А ты любишь петь новогодние песни?,happy,счастье,Интер+есно! +А т+ы л+юбишь п+еть новог+одние п...,4.080375
11424,../../data/dialogs-ru-emotional-conversations/...,S,Очень! Пение новогодних песен - это всегда вес...,happy,счастье,+Очень! П+ение новог+одних п+есен - +это всегд...,5.846271
11425,../../data/dialogs-ru-emotional-conversations/...,M,Согласна! А ты когда-нибудь участвовала в ново...,happy,счастье,Согл+асна! +А т+ы когд+а-ниб+удь уч+аствовала ...,3.900604
11426,../../data/dialogs-ru-emotional-conversations/...,S,"Да, это одно из моих любимых занятий на праздн...",happy,счастье,"Д+а, +это одн+о +из мо+их люб+имых зан+ятий н+...",5.305583


In [116]:
val_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
1,../../data/dialogs-ru-emotional-conversations/...,D,Всё это настолько несправедливо! Хочется крича...,angry,злость,Вс+ё +это наст+олько несправедл+иво! Х+очется ...,11.926854
2,../../data/dialogs-ru-emotional-conversations/...,D,Это просто кошмар. С такими руководителями раб...,angry,злость,+Это пр+осто кошм+ар. С так+ими руковод+ителям...,12.660479
3,../../data/dialogs-ru-emotional-conversations/...,D,Твоя работа всегда казалась мне такой далёкой ...,angry,злость,Тво+я раб+ота всегд+а каз+алась мн+е так+ой да...,14.474083
5,../../data/dialogs-ru-emotional-conversations/...,D,"Давай попробуем. Мы можем справиться, если буд...",happy,счастье,"Дав+ай попр+обуем. М+ы м+ожем спр+авиться, +ес...",5.762250
6,../../data/dialogs-ru-emotional-conversations/...,D,"Полностью согласен. Всегда интересно узнать, к...",arrogance,высокомерие,П+олностью согл+асен. Всегд+а интер+есно узн+а...,7.434854
...,...,...,...,...,...,...,...
173,../../data/dialogs-ru-emotional-conversations/...,S,Уделяю время своим хобби и отдыху.,whisper,шепот,Удел+яю вр+емя сво+им х+обби +и +отдыху.,2.534500
174,../../data/dialogs-ru-emotional-conversations/...,S,"Иваново, улица Ленина, дом сорок пять.",whisper,шепот,"Ив+аново, +улица Л+енина, д+ом с+орок п+ять.",5.160021
176,../../data/dialogs-ru-emotional-conversations/...,S,"Маша, а как обстоят дела с кишечными инфекциям...",neutral,нейтрально,"М+аша, +а к+ак обсто+ят дел+а с киш+ечными инф...",3.960708
178,../../data/dialogs-ru-emotional-conversations/...,S,"Зарайск, улица Ленина, дом четыреста.",arrogance,высокомерие,"Зар+айск, +улица Л+енина, д+ом чет+ыреста.",4.954458


In [5]:
TRAIN_16KHz_CSV = f"{AUDIO_PATH}/train_16KHz.csv"
VAL_16KHz_CSV = f"{AUDIO_PATH}/val_16KHz.csv"

In [6]:
train_df.to_csv(TRAIN_16KHz_CSV, sep='|', index=False)

NameError: name 'train_df' is not defined

In [ ]:
val_df.to_csv(VAL_16KHz_CSV, sep='|', index=False)

In [7]:
train_df = pd.read_csv(TRAIN_16KHz_CSV, sep="|")
train_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,../../data/dialogs-ru-emotional-conversations/...,D,"Привет, Маша!",happy,счастье,"Прив+ет, М+аша!",1.800000
1,../../data/dialogs-ru-emotional-conversations/...,M,"Привет, Дима! Как дела?",happy,счастье,"Прив+ет, Д+има! К+ак дел+а?",1.998729
2,../../data/dialogs-ru-emotional-conversations/...,D,"Да, погодка сегодня... невероятная. А ты тоже ...",happy,счастье,"Д+а, пог+одка сег+одня... неверо+ятная. +А т+ы...",5.461313
3,../../data/dialogs-ru-emotional-conversations/...,M,"О, да, весна - моё любимое время года! Всё вок...",happy,счастье,"+О, д+а, весн+а - мо+ё люб+имое вр+емя г+ода! ...",6.358125
4,../../data/dialogs-ru-emotional-conversations/...,D,"Согласен, я тоже обожаю весну. Особенно нравит...",happy,счастье,"Согл+асен, +я т+оже обож+аю весн+у. Ос+обенно ...",8.769063
...,...,...,...,...,...,...,...
9681,../../data/dialogs-ru-emotional-conversations/...,M,Интересно! А ты любишь петь новогодние песни?,happy,счастье,Интер+есно! +А т+ы л+юбишь п+еть новог+одние п...,4.080375
9682,../../data/dialogs-ru-emotional-conversations/...,S,Очень! Пение новогодних песен - это всегда вес...,happy,счастье,+Очень! П+ение новог+одних п+есен - +это всегд...,5.846271
9683,../../data/dialogs-ru-emotional-conversations/...,M,Согласна! А ты когда-нибудь участвовала в ново...,happy,счастье,Согл+асна! +А т+ы когд+а-ниб+удь уч+аствовала ...,3.900604
9684,../../data/dialogs-ru-emotional-conversations/...,S,"Да, это одно из моих любимых занятий на праздн...",happy,счастье,"Д+а, +это одн+о +из мо+их люб+имых зан+ятий н+...",5.305583


In [8]:
val_df = pd.read_csv(VAL_16KHz_CSV, sep="|")
val_df

,audio_path,speaker_id,text,emotion,emotion_on_russian,accent_text,duration
0,../../data/dialogs-ru-emotional-conversations/...,D,Всё это настолько несправедливо! Хочется крича...,angry,злость,Вс+ё +это наст+олько несправедл+иво! Х+очется ...,11.926854
1,../../data/dialogs-ru-emotional-conversations/...,D,Это просто кошмар. С такими руководителями раб...,angry,злость,+Это пр+осто кошм+ар. С так+ими руковод+ителям...,12.660479
2,../../data/dialogs-ru-emotional-conversations/...,D,Твоя работа всегда казалась мне такой далёкой ...,angry,злость,Тво+я раб+ота всегд+а каз+алась мн+е так+ой да...,14.474083
3,../../data/dialogs-ru-emotional-conversations/...,D,"Давай попробуем. Мы можем справиться, если буд...",happy,счастье,"Дав+ай попр+обуем. М+ы м+ожем спр+авиться, +ес...",5.762250
4,../../data/dialogs-ru-emotional-conversations/...,D,"Полностью согласен. Всегда интересно узнать, к...",arrogance,высокомерие,П+олностью согл+асен. Всегд+а интер+есно узн+а...,7.434854
...,...,...,...,...,...,...,...
143,../../data/dialogs-ru-emotional-conversations/...,S,Уделяю время своим хобби и отдыху.,whisper,шепот,Удел+яю вр+емя сво+им х+обби +и +отдыху.,2.534500
144,../../data/dialogs-ru-emotional-conversations/...,S,"Иваново, улица Ленина, дом сорок пять.",whisper,шепот,"Ив+аново, +улица Л+енина, д+ом с+орок п+ять.",5.160021
145,../../data/dialogs-ru-emotional-conversations/...,S,"Маша, а как обстоят дела с кишечными инфекциям...",neutral,нейтрально,"М+аша, +а к+ак обсто+ят дел+а с киш+ечными инф...",3.960708
146,../../data/dialogs-ru-emotional-conversations/...,S,"Зарайск, улица Ленина, дом четыреста.",arrogance,высокомерие,"Зар+айск, +улица Л+енина, д+ом чет+ыреста.",4.954458


In [ ]:
resampled_waveform, sr = sf.read(f"{AUDIO_PATH}/{train_df.at[1, 'audio_path']}")
Audio(resampled_waveform, rate=sr)

In [119]:
resampled_waveform, sr = sf.read(f"{AUDIO_PATH}/{val_df.at[1, 'audio_path']}")
Audio(resampled_waveform, rate=sr)

In [9]:
s3tokenizer.available_models()

['speech_tokenizer_v1',
 'speech_tokenizer_v1_25hz',
 'speech_tokenizer_v2_25hz',
 'speech_tokenizer_v3_25hz']

In [ ]:
s3tokenizer.load_model("speech_tokenizer_v2_25hz")

In [9]:
s3tokenizerV2 = S3TokenizerV2("speech_tokenizer_v2_25hz")
type(s3tokenizerV2)

s3tokenizer.model_v2.S3TokenizerV2

In [11]:
TRAIN_TTS_DS_PATH = "../../data/train_final_llm_semantic_tokens_dataset.parquet"
VAL_TTS_DS_PATH = "../../data/val_final_llm_semantic_tokens_dataset.parquet"

In [ ]:
train_tts_ds = []
for row in tqdm(train_df.itertuples()):
    current_audio_path = str(row.audio_path)
    full_path = f'{AUDIO_PATH}/{current_audio_path}'
    audio = s3tokenizer.load_audio(full_path)
    mels = s3tokenizer.log_mel_spectrogram(audio)
    mels, mels_lens = s3tokenizer.padding([mels])
    semantic_codes, semantic_codes_lens = s3tokenizerV2.quantize(mels, mels_lens)
    
    train_tts_ds.append({
        "audio_path": current_audio_path,
        "codes": semantic_codes[0, :semantic_codes_lens[0].item()].numpy().astype(np.int16),
        "text": row.text,
        "emotion": row.emotion_on_russian,
    })

table = pa.Table.from_pylist(train_tts_ds)
pq.write_table(table, TRAIN_TTS_DS_PATH, compression="zstd")

9686it [1:21:38,  1.98it/s]


In [ ]:
val_tts_ds = []
for row in tqdm(val_df.itertuples()):
    current_audio_path = str(row.audio_path)
    full_path = f'{AUDIO_PATH}/{current_audio_path}'
    audio = s3tokenizer.load_audio(full_path)
    mels = s3tokenizer.log_mel_spectrogram(audio)
    mels, mels_lens = s3tokenizer.padding([mels])
    semantic_codes, semantic_codes_lens = s3tokenizerV2.quantize(mels, mels_lens)
    
    val_tts_ds.append({
        "audio_path": current_audio_path,
        "codes": semantic_codes[0, :semantic_codes_lens[0].item()].numpy().astype(np.int16),
        "text": row.text,
        "emotion": row.emotion_on_russian,
    })

table = pa.Table.from_pylist(val_tts_ds)
pq.write_table(table, VAL_TTS_DS_PATH, compression="zstd")

148it [01:19,  1.86it/s]


In [14]:
current_train_run_id = f"qwen3-4b-semantic-tokens-sft-lora-{datetime.now().strftime('%Y.%m.%d-%H:%M:%S')}"
log_dir = f"../../logs/tensorboard/{current_train_run_id}"
os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir)

print(f"📊 TensorBoard логи будут сохраняться в: {log_dir}")
print(f"▶️ Для просмотра выполните: tensorboard --logdir=logs/tensorboard")

📊 TensorBoard логи будут сохраняться в: ../../logs/tensorboard/qwen3-4b-semantic-tokens-sft-lora-2026.10.15-20:52:48
▶️ Для просмотра выполните: tensorboard --logdir=logs/tensorboard


In [15]:
BASE_MODEL_PATH = "../../models/qwen3-4b"

In [ ]:
tokenizer: Qwen2Tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    # "models/qwen3-4b-sft-lora-2026.04.28 14:22:51/checkpoint-150",
    trust_remote_code=True
)

In [ ]:
CODEBOOK_SIZE = 3**8
OUTPUT_EOS_ID = CODEBOOK_SIZE
OUTPUT_FILL_ID = CODEBOOK_SIZE + 1
OUTPUT_VOCAB_SIZE = CODEBOOK_SIZE + 2
IGNORE_INDEX = -100


OUTPUT_TOKEN_TO_ID = {
    **{f'<|speech_{i}|>': i for i in range(CODEBOOK_SIZE)},
    "<|fill|>": OUTPUT_FILL_ID,
    "<|eos|>": OUTPUT_EOS_ID,
}
OUTPUT_ID_TO_TOKEN = dict(
    zip(OUTPUT_TOKEN_TO_ID.values(), OUTPUT_TOKEN_TO_ID.keys())
)

In [ ]:
new_tokens = [
    "<|sos|>",
    "<|turn of speech|>",
    *OUTPUT_TOKEN_TO_ID.keys(),
]
existing = set(tokenizer.get_vocab().keys())
for tok in new_tokens:
    if tok in existing:
        print(f"Уже есть: {tok}")

In [18]:
for target in new_tokens:
    conflicts = [tok for tok in tokenizer.get_vocab() if target in tok and tok != target]
    print(target)
    print(f"Конфликтующих токенов: {len(conflicts)}")
    for tok in conflicts[:20]:
        print(repr(tok))

<|sos|>
Конфликтующих токенов: 0
<|turn of speech|>
Конфликтующих токенов: 0
<|eos|>
Конфликтующих токенов: 0
<|fill|>
Конфликтующих токенов: 0


In [19]:
# 2. Размер до
vocab_before = len(tokenizer)
print(f"Размер словаря до: {vocab_before}")

# 3. Добавление
num_added = tokenizer.add_special_tokens({"additional_special_tokens": new_tokens})
print(f"Добавлено: {num_added}")

# 4. Размер после
vocab_after = len(tokenizer)
print(f"Размер словаря после: {vocab_after}")
assert vocab_after == vocab_before + num_added

# 5. Проверка ID
for tok in new_tokens:
    tok_id = tokenizer.convert_tokens_to_ids(tok)
    assert tok_id != tokenizer.unk_token_id, f"{tok} не добавлен!"
    print(f"{tok} → {tok_id}")

Размер словаря до: 151669
Добавлено: 4
Размер словаря после: 151673
<|sos|> → 151669
<|turn of speech|> → 151670
<|eos|> → 151671
<|fill|> → 151672


In [29]:
new_token_ids = []
for tok in new_tokens:
    tok_id = tokenizer.convert_tokens_to_ids(tok)
    new_token_ids.append(tok_id)
new_token_ids

[151669, 151670, 151671, 151672]

In [28]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True,
    attn_implementation="sdpa",
    # attn_implementation="eager",
    # dtype=torch.float16,
    dtype=torch.bfloat16
)
# Размер словаря токенизатора
print("Vocab size (tokenizer):", len(tokenizer))

# Размер эмбеддингов модели
print("Embed size (model):", base_model.get_input_embeddings().weight.shape[0]) # type: ignore


base_model.resize_token_embeddings(max(len(tokenizer), base_model.model.embed_tokens.num_embeddings))
print(f"Эмбеддинги расширены до: {base_model.get_input_embeddings().weight.shape[0]}") # type: ignore

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Vocab size (tokenizer): 151673
Embed size (model): 151936
Эмбеддинги расширены до: 151936


In [21]:
base_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151673, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [ ]:
powers = torch.pow(3, torch.arange(2**3))
h = torch.Tensor([2]*8)
mu = torch.sum(h * powers.unsqueeze(0), dim=-1)
mu # the biggest value => size is 6561 because of 0

tensor([6560.])

In [59]:
3**8

6561

In [1]:
result = 0
for i in range(8):
    result += 2 * (3**i)
print(result)

6560


In [1]:
hidden_size = 2560
new_vocab_size = 3**8 + 2 # [0,6560] + {fill, eos}
new_head = nn.Linear(hidden_size, new_vocab_size, bias=False)
base_model.lm_head = new_head

NameError: name 'nn' is not defined

In [ ]:
base_model.config.tie_word_embeddings = False
base_model.config.use_cache = False

In [25]:
print(base_model.config.tie_word_embeddings)
print(base_model.lm_head.weight.data_ptr() == base_model.model.embed_tokens.weight.data_ptr())

True
False


In [65]:
tokenizer.pad_token_id

151643

In [ ]:
@dataclass
class TTSDatasetItem:
    input_ids: torch.Tensor
    labels: torch.Tensor
    padding_mask: torch.Tensor


@dataclass
class TTSDataItem:
    audio_path: str
    codes: torch.Tensor
    text: str
    emotion: str


@dataclass
class OutputTokenizerResult:
    labels: torch.Tensor


def output_tokenizer(label_list: list[str|int]) -> OutputTokenizerResult:
    try:
        ids = [OUTPUT_TOKEN_TO_ID[token] if token != IGNORE_INDEX else IGNORE_INDEX for token in label_list] # type: ignore
    except KeyError:
        raise ValueError("Semantic code is outside the configured output vocabulary")
    return OutputTokenizerResult(torch.tensor(ids, dtype=torch.long))


class PovaryoshkaTTSDataset(Dataset):
    def __init__(
        self,
        data: list[TTSDataItem],
        N: int,
        M: int,
        input_tokenizer: Qwen2Tokenizer,
        output_tokenizer: Callable[[list[str|int]], OutputTokenizerResult]
    ):
        self.__data = data
        self.__N = N
        self.__M = M
        self.__input_tokenizer = input_tokenizer
        self.__output_tokenizer = output_tokenizer
    
    def __len__(self):
        return len(self.__data)

    def __getitem__(self, index: int) -> TTSDatasetItem:
        # a b c d 1 2 3 4 5 f a b c t 6 7 8        <- input 
        # v v v 1 2 3 4 5 f v v v v 6 7 8 e        <- output
        item = self.__data[index]
        tokenized_text = self.__input_tokenizer.tokenize(f'<{item.emotion}>{item.text}</{item.emotion}>')
        new_tokenized_text: list[str] = ["<|sos|>"]
        str_semantic_codes = [OUTPUT_ID_TO_TOKEN[item] for item in item.codes.tolist()]
        label_list: list[str|int] = [IGNORE_INDEX]
        chunk_size = self.__N
        # To avoid model synthesis of the whole mel-spectrogram before getting all codes
        if len(str_semantic_codes) * self.__N <= len(tokenized_text) * self.__M:
            chunk_size = len(tokenized_text)
        j = 0
        for i in range(0, len(tokenized_text), chunk_size):
            new_token_block = tokenized_text[i:i + chunk_size]
            if i + chunk_size >= len(tokenized_text):
                new_token_block.append("<|turn of speech|>")
            new_tokenized_text.extend(new_token_block)
            label_list.extend([IGNORE_INDEX]*(len(new_token_block) - 1))
            new_tokenized_text.extend(str_semantic_codes[j:j + self.__M])
            label_list.extend(str_semantic_codes[j:j + self.__M])
            j += self.__M
            if "<|turn of speech|>" not in new_tokenized_text:
                new_tokenized_text.append("<|fill|>")
                label_list.extend(["<|fill|>", IGNORE_INDEX])
        if "<|turn of speech|>" not in new_tokenized_text:
            new_tokenized_text.append("<|turn of speech|>")
        if j < len(str_semantic_codes):
            new_tokenized_text.extend(str_semantic_codes[j:])
            label_list.extend(str_semantic_codes[j:])
        label_list.append("<|eos|>")
        input_ids = torch.tensor(
            self.__input_tokenizer.convert_tokens_to_ids(new_tokenized_text),
            dtype=torch.long
        )
        output_tokenizer_output = self.__output_tokenizer(label_list)
        return TTSDatasetItem(
            input_ids=input_ids,
            padding_mask=torch.ones_like(input_ids),
            labels=output_tokenizer_output.labels,
        )


def collate_fn(
    batch: list[TTSDatasetItem],
    input_pad_token_id = int(tokenizer.pad_token_id), # type: ignore
    ignore_index = IGNORE_INDEX
) -> dict[str, torch.Tensor]:
    B = len(batch)

    # 1. Максимальные длины в батче
    max_in_len = max(item.input_ids.size(0) for item in batch)
    max_out_len = max(item.labels.size(0) for item in batch)

    # 2. Аллоцируем тензоры нужной формы
    input_ids = torch.full(
        (B, max_in_len), input_pad_token_id, dtype=torch.long
    )
    padding_mask = torch.zeros(
        (B, max_in_len), dtype=torch.long
    )
    labels = torch.full(
        (B, max_out_len), ignore_index, dtype=torch.long
    )

    # 3. Копируем данные
    for i, item in enumerate(batch):
        n_in = item.input_ids.size(0)
        n_out = item.labels.size(0)

        input_ids[i, :n_in] = item.input_ids
        padding_mask[i, :n_in] = item.padding_mask
        labels[i, :n_out] = item.labels

    return {
        "input_ids": input_ids,           # (B, T_in)
        "attention_mask": padding_mask,     # (B, T_in)
        "labels": labels,                 # (B, T_out)
    }


False

In [ ]:
def load_tts_parquet(path: str) -> list[TTSDataItem]:
    df = pd.read_parquet(path)
    items = []
    for row in tqdm(df.itertuples(), total=len(df), desc="Loading items"):
        codes_array = np.array(row.codes, dtype=np.int16)
        items.append(TTSDataItem(
            audio_path=row.audio_path,
            codes=codes_array,
            text=row.text,
            emotion=row.emotion
        ))
    return items

In [ ]:
train_data_items = load_tts_parquet(TRAIN_TTS_DS_PATH)
val_data_items   = load_tts_parquet(VAL_TTS_DS_PATH)

In [ ]:
train_dataset = PovaryoshkaTTSDataset(
    data=train_data_items,
    N=5,
    M=15,
    input_tokenizer=tokenizer,
    output_tokenizer=output_tokenizer
)

In [ ]:
val_dataset = PovaryoshkaTTSDataset(
    data=val_data_items,
    N=5,
    M=15,
    input_tokenizer=tokenizer,
    output_tokenizer=output_tokenizer
)

In [ ]:
import numpy as np
from transformers import TrainerCallback

# ============================================================
# 7. МЕТРИКИ (LOSS И ACCURACY)
# ============================================================

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    speech_mask = (labels >= 0) & (labels < CODEBOOK_SIZE)
    special_mask = (labels == OUTPUT_EOS_ID) | (labels == OUTPUT_FILL_ID)
    result = {}
    for name, mask in [("speech_accuracy", speech_mask), ("special_accuracy", special_mask)]:
        if mask.any():
            result[name] = float(np.mean(predictions[mask] == labels[mask]))
    return result
    

# ============================================================
# 8. КАСТОМНЫЙ КОЛБЭК ДЛЯ TENSORBOARD
# ============================================================
class TensorBoardCallback(TrainerCallback):
    """Логирует метрики обучения в TensorBoard"""
    def __init__(self, writer):
        self.writer = writer

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Логируем train loss
            if "loss" in logs:
                self.writer.add_scalar("train/loss", logs["loss"], state.global_step)
            if "grad_norm" in logs:
                self.writer.add_scalar("train/grad_norm", logs["grad_norm"], state.global_step)
            if "learning_rate" in logs:
                self.writer.add_scalar("train/learning_rate", logs["learning_rate"], state.global_step)
            
            # Логируем eval метрики (eval_loss и eval_accuracy)
            for key, value in logs.items():
                if key.startswith("eval_"):
                    # Переименовываем для красоты (eval_loss -> eval/loss)
                    metric_name = key.replace("eval_", "eval/")
                    self.writer.add_scalar(metric_name, value, state.global_step)

In [ ]:
def tts_loss(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits
    if logits.shape[:-1] != labels.shape:
        raise ValueError("logits and labels have incompatible sequence shapes.")
    total_loss = F.cross_entropy(
        logits.float().reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
        ignore_index=IGNORE_INDEX,
        reduction="sum",
    )
    count = ((labels != IGNORE_INDEX).sum()
             if num_items_in_batch is None else num_items_in_batch)
    count = torch.as_tensor(count, device=logits.device).clamp_min(1)
    return total_loss / count


In [ ]:
# ============================================================
# 9. LORA CONFIG
# ============================================================
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    modules_to_save=["lm_head"],
    trainable_token_indices=new_token_ids
)
# 1. Загружаем конфигурацию чекпоинта
# checkpoint_path = "models/qwen3-4b-sft-lora-2026.04.28-14:22:51/checkpoint-150"
# peft_config = PeftConfig.from_pretrained(checkpoint_path)

# # 2. Загружаем LoRA веса из чекпоинта ПОВЕРХ базовой модели
# model = PeftModel.from_pretrained(
#     base_model,           # ваша базовая модель (уже загружена)
#     checkpoint_path,      # путь к чекпоинту с весами
#     is_trainable=True     # важно для продолжения обучения!
# )
# model.print_trainable_parameters()

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

parameter_groups = []
for adapter in (True, False):
    for decay in (True, False):
        parameters = [
            parameter for name, parameter in model.named_parameters()
            if parameter.requires_grad
            and ("lora_" in name) == adapter
            and (parameter.ndim >= 2) == decay
        ]
        if parameters:
            parameter_groups.append({
                "params": parameters,
                "lr": 1e-4 if adapter else 3e-4,
                "weight_decay": 0.01 if decay else 0.0,
            })
optimizer = torch.optim.AdamW(
    parameter_groups, lr=1e-4, betas=(0.9, 0.999), eps=1e-8, fused=True,
)
# 10. Training arguments: Transformers 4.x and the newer fractional-step API.
if "warmup_ratio" in inspect.signature(TrainingArguments).parameters:
    warmup_kwargs = {"warmup_ratio": 0.03, "warmup_steps": 0}
else:
    warmup_kwargs = {"warmup_steps": 0.03}

# ============================================================
# 10. TRAINING ARGS
# ============================================================
training_args = TrainingArguments(
    # dataset_text_field="messages",
    output_dir=f"models/{current_train_run_id}",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch: 32 on ONE GPU.
    num_train_epochs=5,
    learning_rate=1e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    save_only_model=False,
    eval_accumulation_steps=1,
    bf16=True,
    fp16=False,
    tf32=True,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    label_names=["labels"],
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    seed=42,
    data_seed=42,
    **warmup_kwargs, # type: ignore
)

# ============================================================
# 11. SFT TRAINER
# ============================================================
tensorboard_callback = TensorBoardCallback(writer)
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=2,          # Ждем 5 эпох без улучшения
    early_stopping_threshold=0.001      # Считаем улучшением только изменение loss более чем на 0.001
)
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    compute_loss_func=tts_loss,
    optimizers=(optimizer, None),
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

# ============================================================
# 12. TRAIN
# ============================================================
print("\n" + "="*60)
print("НАЧАЛО ОБУЧЕНИЯ")
print("="*60)
print(f"📊 Для просмотра графиков выполните в другом терминале:")
print(f"   tensorboard --logdir=logs/tensorboard")
print("="*60 + "\n")

# Записываем гиперпараметры в TensorBoard
hyperparams = {
    "model": "qwen3-4b",
    "lora_r": peft_config.r,
    "lora_alpha": peft_config.lora_alpha,
    "learning_rate": training_args.learning_rate,
    "batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation": training_args.gradient_accumulation_steps,
    "epochs": training_args.num_train_epochs,
}
for key, value in hyperparams.items():
    writer.add_text("hyperparams/" + key, str(value), 0)

# trainer.train(resume_from_checkpoint=checkpoint_path)
trainer.train()

# ============================================================
# 13. SAVE
# ============================================================
trainer.save_model( # type: ignore
    f"models/final-model"
)
tokenizer.save_pretrained(
    f"models/final-tokenizer"
)

# Закрываем TensorBoard writer
writer.close()
print(f"\n✅ Обучение завершено! Логи сохранены в: {log_dir}")
print(f"▶️ Для просмотра выполните: tensorboard --logdir=logs/tensorboard")